# Disease Diagnosis from Symptoms: Machine Learning Workflow

This notebook refactors the logic from `clips_diagnose.py` into a machine learning workflow. It uses the same data files and structure, and guides you through data loading, preprocessing, model training, and prediction.

## 1. Import Required Libraries
Import Python libraries for data processing and machine learning.

In [13]:
# Import Required Libraries
import os
import csv
import unicodedata
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 2. Set Up Data Paths
Define the paths to the data and translations directories, matching the structure used in `clips_diagnose.py`.

In [14]:
# Set Up Data Paths
script_dir = os.path.abspath(os.getcwd())
data_path = os.path.join(script_dir, 'data')
csv_path = os.path.join(data_path, 'disease-symptoms-augmented.csv')
translations_path = os.path.join(data_path, 'translations')
print('Data path:', data_path)
print('CSV path:', csv_path)
print('Translations path:', translations_path)

Data path: c:\Users\MSI\OneDrive\Bureau\PFE\PFE\Server\data
CSV path: c:\Users\MSI\OneDrive\Bureau\PFE\PFE\Server\data\disease-symptoms-augmented.csv
Translations path: c:\Users\MSI\OneDrive\Bureau\PFE\PFE\Server\data\translations


## 3. Normalize Strings Function
Implement the `normalize_string` function to preprocess disease and symptom names as in the original code.

In [15]:
# Normalize Strings Function
def normalize_string(s):
    if any('\u0600' <= c <= '\u06FF' for c in s):  # Detect Arabic characters
        return s.strip().replace(" ", "_").lower()
    return unicodedata.normalize('NFD', s).encode('ASCII', 'ignore').decode('ASCII').strip().replace(" ", "_").lower()

## 4. Load Disease Information from CSV
Load disease descriptions and precautions from the appropriate CSV files for a given language.

In [16]:
# Load Disease Information from CSV
def load_disease_info(lang='en'):
    if lang == 'en':
        desc_path = os.path.join(data_path, 'disease-description.csv')
        prec_path = os.path.join(data_path, 'disease-precaution.csv')
    else:
        desc_path = os.path.join(translations_path, f'disease-description_{lang}.csv')
        prec_path = os.path.join(translations_path, f'disease-precaution_{lang}.csv')
    descriptions = {}
    precautions = {}
    with open(desc_path, "r", encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        disease_col = next((col for col in reader.fieldnames if col.lower() in ['disease', 'maladie', 'مرض']), None)
        for row in reader:
            disease_key = normalize_string(row[disease_col])
            descriptions[disease_key] = row.get('description', "Description non disponible.").strip()
    with open(prec_path, "r", encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        disease_col = next((col for col in reader.fieldnames if col.lower() in ['disease', 'maladie', 'مرض']), None)
        for row in reader:
            disease_key = normalize_string(row[disease_col])
            precs = [row.get(f'precaution_{i}', '').strip().capitalize() for i in range(1, 5) if row.get(f'precaution_{i}', '').strip()]
            precautions[disease_key] = precs if precs else ["Précaution non disponible."]
    return descriptions, precautions

## 5. Load Symptom-Disease Mapping from CSV
Load the mapping between diseases and their symptoms from the CSV files, normalizing all entries.

In [17]:
# Load Symptom-Disease Mapping from CSV
def load_symptom_disease_mapping(lang='en'):
    symptom_disease_map = {}
    if lang == 'en':
        mapping_path = os.path.join(data_path, 'disease-symptoms.csv')
    else:
        mapping_path = os.path.join(translations_path, f'disease-symptoms_{lang}.csv')
    with open(mapping_path, "r", encoding='utf-8-sig', errors='replace') as f:
        reader = csv.DictReader(f)
        disease_col = next((col for col in reader.fieldnames if col.lower() in ['disease', 'maladie', 'مرض']), None)
        for row in reader:
            if not row or not row[disease_col]:
                continue
            disease = normalize_string(row[disease_col])
            symptom_prefix = 'الأعراض_' if lang == 'ar' else 'Symptom_' if lang == 'en' else 'Symptome_'
            symptoms = []
            for j in range(1, 18):
                symptom_key = f'{symptom_prefix}{j}'
                symptom_value = row.get(symptom_key, None)
                if symptom_value is not None:
                    normalized = normalize_string(symptom_value)
                    if normalized:
                        symptoms.append(normalized)
            if len(symptoms) < 2:
                continue
            key_symptoms = symptoms[:2]
            optional_symptoms = symptoms[2:] if len(symptoms) > 2 else []
            symptom_disease_map[disease] = {'key': key_symptoms, 'optional': optional_symptoms}
    return symptom_disease_map

## 6. Prepare Dataset for Machine Learning
Transform the symptom-disease mapping into a format suitable for machine learning, such as a multi-hot encoded symptom matrix with disease labels.

In [18]:
# Load CSV Data and Prepare Dataset for Machine Learning
import pandas as pd
df = pd.read_csv(csv_path)
df = df.fillna('')
df['Disease'] = df['Disease'].str.strip().str.lower().str.replace(' ', '_')
for col in df.columns:
    if col.startswith('Symptom_'):
        df[col] = df[col].str.strip().str.lower().str.replace(' ', '_')

# Collect all unique symptoms
all_symptoms = set()
for i in range(1, 18):
    all_symptoms.update(df[f'Symptom_{i}'].unique())
all_symptoms.discard('')
all_symptoms = sorted(list(all_symptoms))

def row_to_vector(row):
    symptoms = set(row[f'Symptom_{i}'] for i in range(1, 18) if row[f'Symptom_{i}'])
    return [1 if s in symptoms else 0 for s in all_symptoms]

X = np.array([row_to_vector(row) for _, row in df.iterrows()])
y = np.array(df['Disease'])
print(f"Number of diseases: {len(set(y))}")
print(f"Number of unique symptoms: {len(all_symptoms)}")

Number of diseases: 7
Number of unique symptoms: 28


## 7. Encode Symptoms and Diseases
Encode symptoms and diseases into numerical representations using techniques like LabelEncoder or MultiLabelBinarizer.

In [19]:
# Encode Symptoms and Diseases
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Example disease labels:', list(le.classes_)[:5])
print('Encoded labels:', y_encoded[:5])

Example disease labels: ['aids', 'allergy', 'chronic_cholestasis', 'drug_reaction', 'fungal_infection']
Encoded labels: [4 4 4 4 4]


## 8. Split Data into Training and Test Sets
Split the dataset into training and test sets to evaluate model performance.

In [20]:
# Split Data into Training and Test Sets (no stratification due to single-sample classes)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

Training samples: 28, Test samples: 7


## 9. Train a Machine Learning Model
Train a classifier (e.g., RandomForestClassifier) to predict diseases from symptoms.

In [21]:
# Train a Machine Learning Model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


## 10. Evaluate Model Performance
Evaluate the trained model using metrics such as accuracy, precision, recall, and confusion matrix.

In [22]:
# Encode and split data, train and evaluate
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
print("Model training complete.")
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.2f}")
unique_labels = np.unique(y_test)
target_names = le.inverse_transform(unique_labels)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=unique_labels, target_names=target_names))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=unique_labels))

Training samples: 28, Test samples: 7
Model training complete.
Test Accuracy: 1.00

Classification Report:
                      precision    recall  f1-score   support

                aids       1.00      1.00      1.00         1
             allergy       1.00      1.00      1.00         1
 chronic_cholestasis       1.00      1.00      1.00         1
       drug_reaction       1.00      1.00      1.00         1
    fungal_infection       1.00      1.00      1.00         1
                gerd       1.00      1.00      1.00         1
peptic_ulcer_disease       1.00      1.00      1.00         1

            accuracy                           1.00         7
           macro avg       1.00      1.00      1.00         7
        weighted avg       1.00      1.00      1.00         7


Confusion Matrix:
[[1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]]


## 11. Predict Disease from Input Symptoms
Create a function to predict the most likely disease(s) given a list of symptoms, and display the corresponding description and precautions.

In [23]:
# Predict Disease from Input Symptoms
def predict_disease(input_symptoms, lang='en'):
    # Normalize input symptoms
    input_symptoms = [normalize_string(s) for s in input_symptoms]
    # Create symptom vector
    symptom_vector = np.array([[1 if s in input_symptoms else 0 for s in all_symptoms]])
    pred_label = clf.predict(symptom_vector)[0]
    disease = le.inverse_transform([pred_label])[0]
    descriptions, precautions = load_disease_info(lang)
    # Normalize disease key for lookup
    disease_key = normalize_string(disease)
    disease_name = disease.replace('_', ' ').title()
    description = descriptions.get(disease_key, 'Description not available.')
    precaution = precautions.get(disease_key, ['Precaution not available.'])
    print(f"Predicted Disease: {disease_name}\nDescription: {description}\nPrecautions: {precaution}")
    return disease_name, description, precaution

# Example usage:
predict_disease(['itching', 'skin_rash', 'abdominal_pain'])

Predicted Disease: Fungal Infection
Description: In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. Fungi can live in the air, soil, water, and plants. There are also some fungi that live naturally in the human body. Like many microbes, there are helpful fungi and harmful fungi.
Precautions: ['Bath twice', 'Use detol or neem in bathing water', 'Keep infected area dry', 'Use clean cloths']


('Fungal Infection',
 'In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. Fungi can live in the air, soil, water, and plants. There are also some fungi that live naturally in the human body. Like many microbes, there are helpful fungi and harmful fungi.',
 ['Bath twice',
  'Use detol or neem in bathing water',
  'Keep infected area dry',
  'Use clean cloths'])

In [24]:
# Reload data and test prediction after fixing disease name typos
df = pd.read_csv(csv_path)
df = df.fillna('')
df['Disease'] = df['Disease'].str.strip().str.lower().str.replace(' ', '_')
for col in df.columns:
    if col.startswith('Symptom_'):
        df[col] = df[col].str.strip().str.lower().str.replace(' ', '_')

# Rebuild all_symptoms and model
all_symptoms = set()
for i in range(1, 18):
    all_symptoms.update(df[f'Symptom_{i}'].unique())
all_symptoms.discard('')
all_symptoms = sorted(list(all_symptoms))

def row_to_vector(row):
    symptoms = set(row[f'Symptom_{i}'] for i in range(1, 18) if row[f'Symptom_{i}'])
    return [1 if s in symptoms else 0 for s in all_symptoms]

X = np.array([row_to_vector(row) for _, row in df.iterrows()])
y = np.array(df['Disease'])
le = LabelEncoder()
y_encoded = le.fit_transform(y)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y_encoded)

# Test prediction again
predict_disease(['continuous_sneezing', 'shivering', 'watering_from_eyes'])

Predicted Disease: Allergy
Description: An allergy is an immune system response to a foreign substance that's not typically harmful to your body.They can include certain foods, pollen, or pet dander. Your immune system's job is to keep you healthy by fighting harmful pathogens.
Precautions: ['Apply calamine', 'Cover area with bandage', 'Use ice to compress itching']


('Allergy',
 "An allergy is an immune system response to a foreign substance that's not typically harmful to your body.They can include certain foods, pollen, or pet dander. Your immune system's job is to keep you healthy by fighting harmful pathogens.",
 ['Apply calamine', 'Cover area with bandage', 'Use ice to compress itching'])